# Notebook 5: Larger Embeddings: `all-mpnet-base-v2`
**Swap from 384-dim to 768-dim embeddings and compare search quality**

The core notebooks use `all-MiniLM-L6-v2` (384 dims, 22M params) for speed.
In this notebook we upgrade to `all-mpnet-base-v2` (768 dims, 109M params). The
top-performing all-round sentence-transformer and compare retrieval quality.

| Model | Dims | Params | STS Benchmark |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 22M | 82.0 |
| `all-mpnet-base-v2` | 768 | 109M | 83.8 |

### What changes
- New OpenSearch index with **768-dim** `knn_vector` field
- Re-embed all chunks with the larger model
- Compare search results side-by-side

### Prerequisites
- Notebook 01 completed (the 384-dim index exists for comparison)
- OpenSearch running

## 0 · Setup

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from sentence_transformers import SentenceTransformer
from src.opensearch_client import (
    get_client, create_index_custom, bulk_index,
    knn_search, INDEX_NAME,
)
from src.parser import load_podcast_chunks
import numpy as np

client = get_client()
DATA_DIR = os.path.join(os.getcwd(), '..', 'vector-podcast')

SMALL_INDEX = INDEX_NAME              # 384-dim (from notebook 01)
LARGE_INDEX = 'vector-podcast-768'    # 768-dim (we create below)

print(f'✅ OpenSearch connected')
print(f'   Existing 384-dim index: {client.count(index=SMALL_INDEX)["count"]} chunks')

## 1 · Load Both Models

In [ ]:
print('Loading small model (all-MiniLM-L6-v2)...')
model_small = SentenceTransformer('all-MiniLM-L6-v2')
print(f'  → {model_small.get_sentence_embedding_dimension()} dims')

print('Loading large model (all-mpnet-base-v2, ~420 MB download on first run)...')
model_large = SentenceTransformer('all-mpnet-base-v2')
print(f'  → {model_large.get_sentence_embedding_dimension()} dims')

## 2 · Create 768-dim Index & Embed

We create a separate OpenSearch index (`vector-podcast-768`) so we can query both and compare.

In [ ]:
# Create the 768-dim index
create_index_custom(client, LARGE_INDEX, embedding_dim=768, recreate=False)

# Load and chunk podcast data (same chunking as notebook 01)
chunks = load_podcast_chunks(DATA_DIR, chunk_size=400, overlap=50)
print(f'Loaded {len(chunks)} chunks')

In [ ]:
# Embed with the larger model
texts = [c.chunk_text for c in chunks]

print(f'Embedding {len(texts)} chunks with all-mpnet-base-v2 (this takes a bit longer)...')
t0 = time.time()
embeddings_large = model_large.encode(
    texts, batch_size=32, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
t_large = time.time() - t0

print(f'\n✅ Done in {t_large:.1f}s. Shape: {embeddings_large.shape}')

In [ ]:
# Bulk-index into the 768-dim index
from src.opensearch_client import bulk_index as _bulk_index
from opensearchpy import helpers

def bulk_index_custom(client, index_name, chunks, embeddings, batch_size=100):
    def _actions():
        for chunk, emb in zip(chunks, embeddings):
            yield {
                '_index': index_name,
                '_id': chunk.doc_id,
                '_source': {
                    'embedding': emb,
                    'episode_title': chunk.episode_title,
                    'chunk_text': chunk.chunk_text,
                    'url': chunk.url,
                    'pub_date': chunk.pub_date,
                    'description': chunk.description,
                    'chunk_index': chunk.chunk_index,
                    'total_chunks': chunk.total_chunks,
                },
            }
    success, _ = helpers.bulk(client, _actions(), chunk_size=batch_size, request_timeout=60)
    return success

print('Indexing 768-dim embeddings...')
indexed = bulk_index_custom(client, LARGE_INDEX, chunks, embeddings_large.tolist())
client.indices.refresh(LARGE_INDEX)
print(f'✅ Indexed {indexed} documents into {LARGE_INDEX}')

## 3 · Side-by-Side Search Comparison

Same queries, two indices. Let's see if 768 dims finds better results.

In [ ]:
def compare_search(query: str, k: int = 3):
    """Search both indices and print results side by side."""
    emb_small = model_small.encode(query, normalize_embeddings=True).tolist()
    emb_large = model_large.encode(query, normalize_embeddings=True).tolist()

    results_384 = knn_search(client, emb_small, k=k)

    # Search the 768-dim index
    knn_query = {'vector': emb_large, 'k': k}
    resp = client.search(
        index=LARGE_INDEX,
        body={'size': k, 'query': {'knn': {'embedding': knn_query}}}
    )
    results_768 = [
        {
            'score': hit['_score'],
            'episode_title': hit['_source']['episode_title'],
            'chunk_text': hit['_source']['chunk_text'],
        }
        for hit in resp['hits']['hits']
    ]

    print(f'Query: "{query}"')
    print()
    print(f'{"384-dim (MiniLM)":<45s} {"768-dim (MPNet)":<45s}')
    print('─' * 45 + ' ' + '─' * 45)
    for i in range(k):
        left = f"[{results_384[i]['score']:.3f}] {results_384[i]['episode_title'][:35]}" if i < len(results_384) else ''
        right = f"[{results_768[i]['score']:.3f}] {results_768[i]['episode_title'][:35]}" if i < len(results_768) else ''
        print(f'{left:<45s} {right:<45s}')
    print()


compare_search('How does HNSW handle high-dimensional data?')
compare_search('What are the economics of running embedding models at scale?')
compare_search('How to measure search relevance?')
compare_search('Sparse vs dense vectors for hybrid search')

## 4 · Embedding Speed Comparison

Larger models produce better embeddings but take longer to encode.

In [ ]:
sample_texts = texts[:100]

t0 = time.time()
model_small.encode(sample_texts, batch_size=64, normalize_embeddings=True)
t_small = time.time() - t0

t0 = time.time()
model_large.encode(sample_texts, batch_size=32, normalize_embeddings=True)
t_large = time.time() - t0

print(f'Encoding 100 chunks on CPU:')
print(f'  all-MiniLM-L6-v2 (384d):  {t_small:.2f}s  ({100/t_small:.0f} chunks/sec)')
print(f'  all-mpnet-base-v2 (768d):  {t_large:.2f}s  ({100/t_large:.0f} chunks/sec)')
print(f'  Slowdown factor: {t_large/t_small:.1f}x')

## 5 · Index Size Comparison

In [ ]:
for idx_name in [SMALL_INDEX, LARGE_INDEX]:
    stats = client.indices.stats(index=idx_name)
    size_bytes = stats['indices'][idx_name]['primaries']['store']['size_in_bytes']
    doc_count = stats['indices'][idx_name]['primaries']['docs']['count']
    size_mb = size_bytes / (1024 * 1024)
    print(f'{idx_name:30s}  {doc_count:5d} docs  {size_mb:8.1f} MB')

## Key Takeaways

| Dimension | `all-MiniLM-L6-v2` (384d) | `all-mpnet-base-v2` (768d) |
|---|---|---|
| Quality (STS) | 82.0 | **83.8** |
| Encoding speed | ⚡ Fast | Slower (~3–5x) |
| Index size | Smaller | ~2x larger |
| RAM usage | Lower | Higher |

**When to upgrade:**
- Queries that require **nuanced semantic understanding** (e.g. paraphrased content)
- You have GPU or enough CPU budget for encoding
- Index size and search latency are not bottlenecks

**When to stay small:**
- Workshop / prototyping
- Large-scale indices (millions of vectors) on limited hardware
- Latency-critical real-time search

### Cleanup (optional)
```python
# Delete the 768-dim index if you don't need it
client.indices.delete('vector-podcast-768')
```